
# 04 – Iteration 3: Dataset Preparation

This notebook:

1. Loads the feature table and selected features list  
2. Removes leakage features (anomaly codes and flags)  
3. Performs **group-aware** train/valid/test splitting by meter (`NUMEROSERIECONTADOR`)  
4. Uses **5% stratified samples** for train/valid/test to keep memory usage under control  
5. Fits preprocessing (Median Imputer + Standard Scaler) on the sampled training set only  
6. Applies preprocessing to the sampled splits  
7. Saves all prepared matrices and preprocessing params under `iteration_3/results/prepared/`


In [1]:

import os
import sys
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Allow importing from src/
SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

from splitting import make_group_splits, add_split_column
from preprocessing import fit_preprocessor, apply_preprocessor, save_preprocessor

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

FEATURES_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "results", "features"))
SELECTED_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "results", "selected"))
PREPARED_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "results", "prepared"))
os.makedirs(PREPARED_DIR, exist_ok=True)

FEATURES_PARQUET_PATH = os.path.join(FEATURES_DIR, "features_dataset2_iter3.parquet")
SELECTED_FEATURES_PATH = os.path.join(SELECTED_DIR, "selected_features_iter3.txt")

print("FEATURES_PARQUET_PATH:", FEATURES_PARQUET_PATH)
print("SELECTED_FEATURES_PATH:", SELECTED_FEATURES_PATH)
print("PREPARED_DIR:", PREPARED_DIR)


FEATURES_PARQUET_PATH: c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\features\features_dataset2_iter3.parquet
SELECTED_FEATURES_PATH: c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\selected\selected_features_iter3.txt
PREPARED_DIR: c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\prepared


In [2]:

# 1) Load feature table
feat_df = pd.read_parquet(FEATURES_PARQUET_PATH)
print("Loaded features. Shape:", feat_df.shape)
display(feat_df.head())

# 2) Load selected feature names
with open(SELECTED_FEATURES_PATH, "r") as f:
    selected_features = [line.strip() for line in f if line.strip()]

print("Selected features from file:", selected_features)


Loaded features. Shape: (21195970, 19)


,POLISSA_SUBM,CODI_ANOMALIA,START_DATE,END_DATE,US_AIGUA_SUBM,SECCIO_CENSAL,NUMEROSERIECONTADOR,CONSUMO_REAL,FECHA_HORA,flag_anom_32768,flag_anom_163840,y_anom,datetime,cons_lag1,delta1,meter_mean,meter_std,cons_z_meter,period_hours
0,UXOZJTJWD74S44CF,163840,2023-09-07,2023-11-08,DOMÈSTIC,0801906022,01172604,0.0,2024-01-01 00:36:20,False,True,1,2024-01-01 00:36:20,NaN,NaN,0.0,0.0,NaN,1488.0
1,UXOZJTJWD74S44CF,163840,2023-09-07,2023-11-08,DOMÈSTIC,0801906022,01172604,0.0,2024-01-01 01:36:20,False,True,1,2024-01-01 01:36:20,0.0,0.0,0.0,0.0,NaN,1488.0
2,UXOZJTJWD74S44CF,163840,2023-09-07,2023-11-08,DOMÈSTIC,0801906022,01172604,0.0,2024-01-01 02:36:32,False,True,1,2024-01-01 02:36:32,0.0,0.0,0.0,0.0,NaN,1488.0
3,UXOZJTJWD74S44CF,163840,2023-09-07,2023-11-08,DOMÈSTIC,0801906022,01172604,0.0,2024-01-01 03:36:32,False,True,1,2024-01-01 03:36:32,0.0,0.0,0.0,0.0,NaN,1488.0
4,UXOZJTJWD74S44CF,163840,2023-09-07,2023-11-08,DOMÈSTIC,0801906022,01172604,0.0,2024-01-01 04:36:32,False,True,1,2024-01-01 04:36:32,0.0,0.0,0.0,0.0,NaN,1488.0


Selected features from file: ['CODI_ANOMALIA', 'period_hours', 'meter_std', 'meter_mean', 'flag_anom_163840', 'cons_z_meter', 'flag_anom_32768', 'CONSUMO_REAL', 'cons_lag1', 'delta1']


In [3]:

ID_COL = "NUMEROSERIECONTADOR"
LABEL_COL = "y_anom"

LEAKAGE_COLS = [
    "CODI_ANOMALIA",
    "flag_anom_32768",
    "flag_anom_163840",
]

# Drop leakage columns from the feature dataframe if present
drop_cols = [c for c in LEAKAGE_COLS if c in feat_df.columns]
if drop_cols:
    print("Dropping leakage columns:", drop_cols)
    feat_df = feat_df.drop(columns=drop_cols)

# Ensure none of the leakage columns remain in the selected feature list
selected_features = [f for f in selected_features if f not in LEAKAGE_COLS]
print("Final selected features (after removing leakage):", selected_features)

# Basic sanity checks
assert ID_COL in feat_df.columns, f"ID column {ID_COL} not found"
assert LABEL_COL in feat_df.columns, f"Label column {LABEL_COL} not found"


Dropping leakage columns: ['CODI_ANOMALIA', 'flag_anom_32768', 'flag_anom_163840']
Final selected features (after removing leakage): ['period_hours', 'meter_std', 'meter_mean', 'cons_z_meter', 'CONSUMO_REAL', 'cons_lag1', 'delta1']


In [4]:

# 3) Group-aware train/valid/test split by meter
train_idx, valid_idx, test_idx = make_group_splits(
    df=feat_df,
    id_col=ID_COL,
    train_size=0.70,
    valid_size=0.15,
    random_state=42,
)

feat_df = add_split_column(feat_df, train_idx, valid_idx, test_idx)
print("Split value counts:")
print(feat_df["split"].value_counts())

df_train = feat_df[feat_df["split"] == "train"]
df_valid = feat_df[feat_df["split"] == "valid"]
df_test  = feat_df[feat_df["split"] == "test"]

print("Train shape:", df_train.shape)
print("Valid shape:", df_valid.shape)
print("Test  shape:", df_test.shape)


Split value counts:
split
train    14811416
test      3202540
valid     3182014
Name: count, dtype: int64
Train shape: (14811416, 17)
Valid shape: (3182014, 17)
Test  shape: (3202540, 17)


In [5]:

# 4) Create 5% stratified samples for each split to control memory usage

train_sample_frac = 0.05
valid_sample_frac = 0.05
test_sample_frac = 0.05

# For very imbalanced labels, ensure stratification uses LABEL_COL
df_train_sample, _ = train_test_split(
    df_train,
    train_size=train_sample_frac,
    stratify=df_train[LABEL_COL],
    random_state=42,
)

df_valid_sample, _ = train_test_split(
    df_valid,
    train_size=valid_sample_frac,
    stratify=df_valid[LABEL_COL],
    random_state=42,
)

df_test_sample, _ = train_test_split(
    df_test,
    train_size=test_sample_frac,
    stratify=df_test[LABEL_COL],
    random_state=42,
)

print("Full train shape:", df_train.shape)
print("Sampled train shape:", df_train_sample.shape)
print("Sampled valid shape:", df_valid_sample.shape)
print("Sampled test  shape:", df_test_sample.shape)

print("\nLabel distribution in sampled TRAIN:")
print(df_train_sample[LABEL_COL].value_counts(normalize=True))


Full train shape: (14811416, 17)
Sampled train shape: (740570, 17)
Sampled valid shape: (159100, 17)
Sampled test  shape: (160127, 17)

Label distribution in sampled TRAIN:
y_anom
1    0.999275
0    0.000725
Name: proportion, dtype: float64


In [6]:

# 5) Fit preprocessor (imputer + scaler) on the TRAIN SAMPLE only
imputer, scaler = fit_preprocessor(
    df=df_train_sample,
    feature_cols=selected_features,
)

print("Preprocessor fitted on sampled train set.")


Preprocessor fitted on sampled train set.


In [7]:

# 6) Apply preprocessor to the sampled splits
X_train = apply_preprocessor(df_train_sample, selected_features, imputer, scaler)
y_train = df_train_sample[LABEL_COL].values

X_valid = apply_preprocessor(df_valid_sample, selected_features, imputer, scaler)
y_valid = df_valid_sample[LABEL_COL].values

X_test  = apply_preprocessor(df_test_sample, selected_features, imputer, scaler)
y_test  = df_test_sample[LABEL_COL].values

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("X_test  shape:", X_test.shape)


X_train shape: (740570, 7)
X_valid shape: (159100, 7)
X_test  shape: (160127, 7)


In [8]:

# 7) Save prepared arrays to disk
np.save(os.path.join(PREPARED_DIR, "X_train.npy"), X_train)
np.save(os.path.join(PREPARED_DIR, "X_valid.npy"), X_valid)
np.save(os.path.join(PREPARED_DIR, "X_test.npy"),  X_test)

np.save(os.path.join(PREPARED_DIR, "y_train.npy"), y_train)
np.save(os.path.join(PREPARED_DIR, "y_valid.npy"), y_valid)
np.save(os.path.join(PREPARED_DIR, "y_test.npy"),  y_test)

print("[OK] Saved sampled train/valid/test matrices in:", PREPARED_DIR)


[OK] Saved sampled train/valid/test matrices in: c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\prepared


In [9]:

# 8) Save preprocessor parameters (imputer + scaler + feature list)
save_preprocessor(
    imputer=imputer,
    scaler=scaler,
    feature_cols=selected_features,
    out_dir=PREPARED_DIR,
    base_name="preprocessor_iter3",
)

print("[OK] Saved preprocessor parameters in:", PREPARED_DIR)


[ok] Saved preprocessor params to: c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\prepared\preprocessor_iter3_params.json
[OK] Saved preprocessor parameters in: c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\prepared


In [10]:

print("Final summary:")
print("Selected features:", selected_features)
print("Train rows (sampled):", X_train.shape[0])
print("Valid rows (sampled):", X_valid.shape[0])
print("Test rows  (sampled):", X_test.shape[0])


Final summary:
Selected features: ['period_hours', 'meter_std', 'meter_mean', 'cons_z_meter', 'CONSUMO_REAL', 'cons_lag1', 'delta1']
Train rows (sampled): 740570
Valid rows (sampled): 159100
Test rows  (sampled): 160127
